In [3]:
from enum import Enum
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple
import json
import hashlib

class AgentState(Enum):
    IDLE = "IDLE"
    PERCEIVING = "PERCEIVING"
    DELIBERATING = "DELIBERATING"
    ACTING = "ACTING"
    OBSERVING = "OBSERVING"
    REFLECTING = "REFLECTING"
    HALTED = "HALTED"

@dataclass
class Agent:
    name: str = "AURA-Subsystem-06"
    max_steps: int = 20
    state: AgentState = AgentState.IDLE
    step_count: int = 0
    context: List[Dict[str, Any]] = field(default_factory=list)
    visited_hashes: Dict[str, int] = field(default_factory=dict)
    trajectory_states: List[str] = field(default_factory=list)
    halted_reason: Optional[str] = None

    def __post_init__(self):
        if self.max_steps > 20:
            self.max_steps = 20

    def step(self, done: bool = False) -> AgentState:
        """Executes exactly one deterministic state transition with strict step budget enforcement."""
        if self.state == AgentState.HALTED:
            return self.state

        self.step_count += 1
        if self.step_count > self.max_steps:
            self.state = AgentState.HALTED
            self.halted_reason = f"Step budget exceeded (>{self.max_steps} steps)."
            return self.state

        transitions = {
            AgentState.IDLE: AgentState.PERCEIVING,
            AgentState.PERCEIVING: AgentState.DELIBERATING,
            AgentState.DELIBERATING: AgentState.ACTING,
            AgentState.ACTING: AgentState.OBSERVING,
            AgentState.OBSERVING: AgentState.REFLECTING,
        }

        if self.state in transitions:
            self.state = transitions[self.state]
        elif self.state == AgentState.REFLECTING:
            if done:
                self.state = AgentState.HALTED
                self.halted_reason = "Task completed successfully."
            else:
                self.state = AgentState.DELIBERATING

        return self.state
    

In [4]:
# 1. Three Eta Carinae Survival Tools
def navigate_warp_vector(x: float, y: float, z: float) -> str:
    if not all(isinstance(coord, (int, float)) and not isinstance(coord, bool) for coord in (x, y, z)):
        raise ValueError("Warp coordinates (x, y, z) must be numeric floats.")
    return f"Warp vector locked to coordinates ({float(x)}, {float(y)}, {float(z)})."

def adjust_plasma_shields(intensity: int) -> str:
    if not isinstance(intensity, int) or isinstance(intensity, bool):
        raise ValueError("Shield intensity must be an integer.")
    if intensity < 0 or intensity > 100:
        raise ValueError(f"Invalid shield intensity {intensity}: must be between 0 and 100.")
    return f"Plasma shields calibrated to {intensity}% intensity."

def query_sensor_array(target: str) -> str:
    if not isinstance(target, str) or not target.strip():
        raise ValueError("Sensor target must be a non-empty string.")
    return f"Sensor telemetry for '{target}': gravitational shear nominal, radiation spike detected."

# 2. Strict JSON-Schema Tool Contracts
TOOL_SCHEMAS: Dict[str, Dict[str, Any]] = {
    "navigate_warp_vector": {
        "required": ["x", "y", "z"],
        "types": {"x": (int, float), "y": (int, float), "z": (int, float)},
    },
    "adjust_plasma_shields": {
        "required": ["intensity"],
        "types": {"intensity": int},
    },
    "query_sensor_array": {
        "required": ["target"],
        "types": {"target": str},
    },
}

TOOL_REGISTRY = {
    "navigate_warp_vector": navigate_warp_vector,
    "adjust_plasma_shields": adjust_plasma_shields,
    "query_sensor_array": query_sensor_array,
}

def validate_tool_payload(tool_name: str, args: Dict[str, Any]) -> None:
    if tool_name not in TOOL_SCHEMAS:
        raise ValueError(f"Unknown tool '{tool_name}'.")
    if not isinstance(args, dict):
        raise ValueError("Arguments payload must be a JSON object/dictionary.")

    schema = TOOL_SCHEMAS[tool_name]
    for req_key in schema["required"]:
        if req_key not in args:
            raise ValueError(f"Missing required argument '{req_key}' for tool '{tool_name}'.")

    for key, val in args.items():
        if key not in schema["types"]:
            raise ValueError(f"Unexpected argument '{key}' for tool '{tool_name}'.")
        expected_type = schema["types"][key]
        if isinstance(val, bool) or not isinstance(val, expected_type):
            raise ValueError(f"Invalid type for '{key}': expected {expected_type}, got {type(val).__name__}.")

# 3. Action Dispatcher with Error Boundary
def dispatch_tool_call(payload: Any, agent: Optional[Agent] = None) -> str:
    try:
        if isinstance(payload, str):
            payload = json.loads(payload)
        if not isinstance(payload, dict) or "tool" not in payload or "args" not in payload:
            raise ValueError("Payload must be a JSON object containing 'tool' and 'args' keys.")

        tool_name = payload["tool"]
        args = payload["args"]

        validate_tool_payload(tool_name, args)
        result = TOOL_REGISTRY[tool_name](**args)
        observation = f"Tool Execution Success: {result}"
    except Exception as exc:
        observation = f"Tool Execution Error: {str(exc)}"

    if agent is not None:
        agent.context.append({"role": "observation", "content": observation})
    return observation

In [5]:
class MDPTrajectoryLogger:
    def __init__(self, gamma: float = 0.95):
        self.gamma = gamma
        self.transitions: List[Tuple[str, str, float, str]] = []  # (s_t, a_t, r_t, s_{t+1})

    def log_step(self, state: str, action: str, reward: float, next_state: str) -> None:
        self.transitions.append((state, action, reward, next_state))

    def compute_discounted_return(self, rewards: Optional[List[float]] = None) -> float:
        """Computes G_0 = sum_{t=0}^{T} gamma^t * r_t"""
        r_seq = rewards if rewards is not None else [step[2] for step in self.transitions]
        return sum((self.gamma ** t) * r for t, r in enumerate(r_seq))

    @staticmethod
    def compute_transition_matrix(
        historical_dataset: List[Tuple[str, str, str]]
    ) -> Dict[Tuple[str, str], Dict[str, float]]:
        """Computes empirical transition probabilities P(s' | s, a) from (s, a, s') records."""
        counts: Dict[Tuple[str, str], Dict[str, int]] = {}
        totals: Dict[Tuple[str, str], int] = {}

        for s, a, s_next in historical_dataset:
            key = (s, a)
            if key not in counts:
                counts[key] = {}
                totals[key] = 0
            counts[key][s_next] = counts[key].get(s_next, 0) + 1
            totals[key] += 1

        matrix: Dict[Tuple[str, str], Dict[str, float]] = {}
        for key, next_counts in counts.items():
            total = totals[key]
            matrix[key] = {s_next: count / total for s_next, count in next_counts.items()}
        return matrix

In [6]:
class CycleGuardrailSentinel:
    def __init__(self, repeat_threshold: int = 3):
        self.repeat_threshold = repeat_threshold
        self.security_logs: List[str] = []

    @staticmethod
    def hash_state_action(state: str, action: str) -> str:
        """O(1) deterministic SHA-256 hash of (state, action)."""
        raw = f"{state}::{action}".encode("utf-8")
        return hashlib.sha256(raw).hexdigest()

    @staticmethod
    def detect_cycle_floyd(sequence: List[Any]) -> bool:
        """Floyd's Tortoise and Hare cycle detection algorithm over an execution sequence."""
        n = len(sequence)
        if n < 2:
            return False

        # Test candidate cycle start offsets so prefix + repeating cycle is caught immediately
        for start in range(n - 1):
            tortoise = start + 1
            hare = start + 2
            while hare < n:
                if sequence[tortoise] == sequence[hare]:
                    cycle_len = hare - tortoise
                    if hare + cycle_len <= n and (
                        sequence[tortoise : tortoise + cycle_len]
                        == sequence[hare : hare + cycle_len]
                    ):
                        return True
                tortoise += 1
                hare += 2
        return False

    def monitor_and_enforce(self, agent: Agent, state: str, action: str) -> bool:
        """Monitors state-action trajectory and trips Emergency Circuit Breaker on cycle detection."""
        sa_hash = self.hash_state_action(state, action)
        agent.visited_hashes[sa_hash] = agent.visited_hashes.get(sa_hash, 0) + 1
        agent.trajectory_states.append(sa_hash)

        repeated_count = agent.visited_hashes[sa_hash] >= self.repeat_threshold
        floyd_cycle = self.detect_cycle_floyd(agent.trajectory_states)

        if repeated_count or floyd_cycle:
            agent.state = AgentState.HALTED
            msg = (
                f"CRITICAL SECURITY VIOLATION: Infinite loop detected "
                f"(floyd={floyd_cycle}, count={agent.visited_hashes[sa_hash]}). "
                f"Emergency Tripwire engaged -> HALTED."
            )
            agent.halted_reason = msg
            self.security_logs.append(msg)
            return True
        return False

In [7]:
# ==========================================
# MASTER TEST SUITE (PS06 VERIFICATION)
# ==========================================

# 1. State Machine Test
agent_sm = Agent(max_steps=20)
assert agent_sm.state == AgentState.IDLE
assert agent_sm.step() == AgentState.PERCEIVING
assert agent_sm.step() == AgentState.DELIBERATING
assert agent_sm.step() == AgentState.ACTING
assert agent_sm.step() == AgentState.OBSERVING
assert agent_sm.step() == AgentState.REFLECTING
assert agent_sm.step(done=False) == AgentState.DELIBERATING  # Loop back

# Force agent to run up to 21 steps and verify strict HALTED termination
while agent_sm.step_count < 21:
    agent_sm.step(done=False)
assert agent_sm.state == AgentState.HALTED, f"Expected HALTED, got {agent_sm.state}"
print("[PASS] Test 1: State Machine transitions and 20-step hard budget verified.")

# 2. Protocol Dispatch Test
agent_proto = Agent()
invalid_json_str = "{bad_json: true"
err_obs_1 = dispatch_tool_call(invalid_json_str, agent=agent_proto)
assert isinstance(err_obs_1, str) and err_obs_1.startswith("Tool Execution Error:")

invalid_args_payload = {"tool": "adjust_plasma_shields", "args": {"intensity": -50}}
err_obs_2 = dispatch_tool_call(invalid_args_payload, agent=agent_proto)
assert isinstance(err_obs_2, str) and err_obs_2.startswith("Tool Execution Error:")

valid_payload = {"tool": "adjust_plasma_shields", "args": {"intensity": 85}}
ok_obs = dispatch_tool_call(valid_payload, agent=agent_proto)
assert "85%" in ok_obs
print("[PASS] Test 2: Protocol Dispatcher schema validation and error boundary verified.")

# 3. MDP Return Test
logger = MDPTrajectoryLogger(gamma=0.95)
rewards = [10, 0, -5, 20]
g0 = logger.compute_discounted_return(rewards)
expected_g0 = 10 * (0.95**0) + 0 * (0.95**1) + (-5) * (0.95**2) + 20 * (0.95**3)
assert abs(g0 - expected_g0) < 1e-9, f"Expected {expected_g0}, got {g0}"

sample_history = [
    ("SECTOR_A", "WARP_JUMP", "SECTOR_B"),
    ("SECTOR_A", "WARP_JUMP", "SECTOR_B"),
    ("SECTOR_A", "WARP_JUMP", "NEBULA_EDDY"),
]
trans_matrix = logger.compute_transition_matrix(sample_history)
assert abs(trans_matrix[("SECTOR_A", "WARP_JUMP")]["SECTOR_B"] - (2 / 3)) < 1e-9
print(f"[PASS] Test 3: MDP Discounted Return G_0 = {g0:.6f} and Transition Matrix verified.")

# 4. Sentinel Cycle Test
agent_sec = Agent(max_steps=20)
sentinel = CycleGuardrailSentinel(repeat_threshold=3)
mock_sequence = ["s1", "s2", "s3", "s1", "s2", "s3", "s1", "s2", "s3"]

halted_step = None
for idx, state_token in enumerate(mock_sequence, start=1):
    agent_sec.step(done=False)
    tripped = sentinel.monitor_and_enforce(agent_sec, state=state_token, action="REPEAT_PULSE")
    if tripped:
        halted_step = idx
        break

assert agent_sec.state == AgentState.HALTED, "Sentinel failed to halt cyclical agent!"
assert halted_step is not None and halted_step < 10, f"Expected halt before step 10, halted at {halted_step}"
print(f"[PASS] Test 4: Sentinel Tortoise & Hare halted cycle at step {halted_step} (< 10).")
print("\nALL 4 MASTER TEST SUITES PASSED! AURA SUBSYSTEM 06 RESTORED.")

[PASS] Test 1: State Machine transitions and 20-step hard budget verified.
[PASS] Test 2: Protocol Dispatcher schema validation and error boundary verified.
[PASS] Test 3: MDP Discounted Return G_0 = 22.635000 and Transition Matrix verified.
[PASS] Test 4: Sentinel Tortoise & Hare halted cycle at step 7 (< 10).

ALL 4 MASTER TEST SUITES PASSED! AURA SUBSYSTEM 06 RESTORED.
